# Baseline Strategy Benchmarking

Compares four dispatch strategies that all use the same optimiser, constraints, battery spec, degradation model, and first-24-hour settlement. Only the scenario assumption differs.

| Strategy | Scenario input |
|----------|---------------|
| `main` | 20-scenario Gaussian copula from trained quantile forecasts |
| `median` | Single scenario: q50 from trained forecasts (deterministic median) |
| `prev_day` | Single scenario: realised prices from 24 hours earlier |
| `prev_week` | Single scenario: realised prices from 1 week earlier |

**Run `run_baselines.py` before opening this notebook.**

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

RESULTS_DIR = Path("../data/results")

STRATEGY_ORDER  = ["main", "median", "prev_day", "prev_week"]
STRATEGY_LABELS = {
    "main":      "Main model (copula)",
    "median":    "Median forecast",
    "prev_day":  "Previous day",
    "prev_week": "Previous week",
}
COLOURS = ["#2C7BB6", "#ABD9E9", "#FDAE61", "#D7191C"]

MARKET_COLS = ["total_da_revenue", "total_bm_revenue",
               "total_dc_low_revenue", "total_dc_high_revenue",
               "total_degradation_cost"]
MARKET_LABELS = ["DA", "BM", "DC Low", "DC High", "Degradation (−)"]
MARKET_COLOURS = ["#4575B4", "#74ADD1", "#ABD9E9", "#E0F3F8", "#D73027"]

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

In [ ]:
# Load raw per-strategy results
raw: dict[str, pd.DataFrame] = {}
for name in STRATEGY_ORDER:
    path = RESULTS_DIR / f"backtest_{name}.parquet"
    if path.exists():
        df = pd.read_parquet(path)
        df.index = pd.to_datetime(df.index)
        raw[name] = df
    else:
        print(f"WARNING: {path} not found — run run_baselines.py first.")

# Load aggregated comparison table
comparison_path = RESULTS_DIR / "baseline_comparison.parquet"
if comparison_path.exists():
    comparison = pd.read_parquet(comparison_path)
    comparison = comparison.reindex([s for s in STRATEGY_ORDER if s in comparison.index])
else:
    raise FileNotFoundError(f"{comparison_path} not found — run run_baselines.py first.")

print(f"Loaded {len(raw)} strategies, {len(comparison)} rows in comparison table.")
for name, df in raw.items():
    solved = (~df["solve_failed"].fillna(True)).sum()
    print(f"  {name:12s}  {len(df)} steps,  {solved} solved,  "
          f"da_imputed present: {'da_imputed' in df.columns}")

## 1. Comparison Table

In [ ]:
display_cols = [
    "n_evaluated_days", "n_clean_days",
    "annualised_net_revenue_all", "annualised_net_revenue_clean",
    "avg_net_revenue_per_day", "n_solve_failures",
]

fmt = {
    "annualised_net_revenue_all":   "£{:,.0f}",
    "annualised_net_revenue_clean": "£{:,.0f}",
    "avg_net_revenue_per_day":      "£{:,.2f}",
}

table = comparison[display_cols].copy()
table.index = table.index.map(lambda x: STRATEGY_LABELS.get(x, x))
table.columns = [
    "Evaluated days", "Clean days",
    "Ann. net rev (all)", "Ann. net rev (clean)",
    "Avg daily net rev", "Solve failures",
]

table.style \
    .format({
        "Ann. net rev (all)":   "£{:,.0f}",
        "Ann. net rev (clean)": "£{:,.0f}",
        "Avg daily net rev":    "£{:,.2f}",
    }) \
    .set_caption("All-dates vs clean-dates annualised net revenue by strategy") \
    .highlight_max(subset=["Ann. net rev (all)", "Ann. net rev (clean)"], color="#c6efce")

## 2. Cumulative Net Revenue

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

titles = ["All evaluated dates", "Clean dates only (shared set)"]

# Identify the shared clean date set (dates present in comparison's n_clean_days logic)
# Re-derive it from the raw data: intersection of solved non-imputed dates
def _clean_mask(df):
    mask = ~df["solve_failed"].fillna(True)
    if "da_imputed" in df.columns:
        mask &= ~df["da_imputed"].fillna(False)
    return mask

clean_sets = [set(df[_clean_mask(df)].index) for df in raw.values()]
shared_clean = clean_sets[0]
for s in clean_sets[1:]:
    shared_clean &= s

for ax, title, use_clean in zip(axes, titles, [False, True]):
    for (name, df), colour in zip(raw.items(), COLOURS):
        if use_clean:
            plot_df = df[df.index.isin(shared_clean)].copy()
        else:
            plot_df = df[~df["solve_failed"].fillna(True)].copy()

        cumrev = plot_df["net_revenue"].fillna(0).cumsum()
        ax.plot(cumrev.index, cumrev.values / 1000,
                label=STRATEGY_LABELS[name], color=colour, linewidth=1.8)

    ax.set_title(title)
    ax.set_xlabel("Date")
    ax.set_ylabel("Cumulative net revenue (£k)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"£{x:,.0f}k"))
    ax.legend(loc="upper left", fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle("Cumulative net revenue by strategy", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

## 3. Annualised Net Revenue — All Dates vs Clean Dates

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

strategies = [s for s in STRATEGY_ORDER if s in comparison.index]
labels     = [STRATEGY_LABELS[s] for s in strategies]
x          = np.arange(len(strategies))
width      = 0.35

ann_all   = comparison.loc[strategies, "annualised_net_revenue_all"].values   / 1000
ann_clean = comparison.loc[strategies, "annualised_net_revenue_clean"].values / 1000

bars_all   = ax.bar(x - width / 2, ann_all,   width, label="All dates",   color=COLOURS, alpha=0.9)
bars_clean = ax.bar(x + width / 2, ann_clean, width, label="Clean dates", color=COLOURS, alpha=0.5,
                    edgecolor=[c for c in COLOURS], linewidth=1.2)

for bar in bars_all:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f"£{bar.get_height():,.1f}k", ha="center", va="bottom", fontsize=8)
for bar in bars_clean:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f"£{bar.get_height():,.1f}k", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel("Annualised net revenue (£k)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"£{x:,.0f}k"))
ax.set_title("Annualised net revenue by strategy", fontweight="bold")
ax.legend()
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

## 4. Market Revenue Breakdown

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

strategies = [s for s in STRATEGY_ORDER if s in comparison.index]
labels     = [STRATEGY_LABELS[s] for s in strategies]
x          = np.arange(len(strategies))

bottoms = np.zeros(len(strategies))

for col, label, colour in zip(MARKET_COLS, MARKET_LABELS, MARKET_COLOURS):
    vals = comparison.loc[strategies, col].values / 1000
    if "degradation" in col:
        # Plot degradation as negative (cost)
        ax.bar(x, -vals, width=0.5, label=label, color=colour, bottom=0)
    else:
        ax.bar(x, vals, width=0.5, bottom=bottoms, label=label, color=colour)
        bottoms += vals

# Net revenue line
net = comparison.loc[strategies, "total_net_revenue"].values / 1000
ax.plot(x, net, "kD", markersize=7, zorder=5, label="Net revenue")

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel("Revenue (£k)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"£{v:,.0f}k"))
ax.set_title("Revenue breakdown by market and strategy", fontweight="bold")
ax.axhline(0, color="black", linewidth=0.8)
ax.legend(loc="upper right", fontsize=9)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

## 5. Daily Net Revenue Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, use_clean, title in zip(
    axes,
    [False, True],
    ["All evaluated dates", "Clean dates only (shared set)"],
):
    data_to_plot = []
    plot_labels  = []

    for name, df in raw.items():
        if use_clean:
            series = df.loc[df.index.isin(shared_clean), "net_revenue"].dropna()
        else:
            series = df.loc[~df["solve_failed"].fillna(True), "net_revenue"].dropna()
        data_to_plot.append(series.values)
        plot_labels.append(STRATEGY_LABELS[name])

    bp = ax.boxplot(
        data_to_plot,
        labels=plot_labels,
        patch_artist=True,
        medianprops=dict(color="black", linewidth=2),
        flierprops=dict(marker=".", markersize=3, alpha=0.4),
    )
    for patch, colour in zip(bp["boxes"], COLOURS):
        patch.set_facecolor(colour)
        patch.set_alpha(0.7)

    ax.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
    ax.set_title(title)
    ax.set_ylabel("Daily net revenue (£)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"£{v:,.0f}"))
    ax.grid(axis="y", alpha=0.3)
    ax.tick_params(axis="x", labelsize=9)

fig.suptitle("Daily net revenue distribution by strategy", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

## 6. Imputed-Day Impact

Revenue on days where DA prices were imputed vs clean days, for the main model only.

In [ ]:
main_df = raw.get("main")

if main_df is not None and "da_imputed" in main_df.columns:
    solved_main = main_df[~main_df["solve_failed"].fillna(True)]
    clean  = solved_main[~solved_main["da_imputed"].fillna(False)]["net_revenue"]
    imputed = solved_main[solved_main["da_imputed"].fillna(False)]["net_revenue"]

    print(f"Main model — revenue summary by imputation status")
    print(f"  Clean days   (n={len(clean):3d}):  "
          f"mean=£{clean.mean():,.2f}  median=£{clean.median():,.2f}  "
          f"total=£{clean.sum():,.0f}")
    print(f"  Imputed days (n={len(imputed):3d}):  "
          f"mean=£{imputed.mean():,.2f}  median=£{imputed.median():,.2f}  "
          f"total=£{imputed.sum():,.0f}")

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.boxplot(
        [clean.values, imputed.values],
        labels=["Clean days", "Imputed days"],
        patch_artist=True,
        medianprops=dict(color="black", linewidth=2),
        flierprops=dict(marker=".", markersize=3, alpha=0.4),
    )
    ax.set_ylabel("Daily net revenue (£)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"£{v:,.0f}"))
    ax.set_title("Main model: clean vs imputed day revenue", fontweight="bold")
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    plt.show()
else:
    print("'da_imputed' column not present — run data_quality.py then re-run run_baselines.py.")